#### RMSNorm
- Imagine one token has this embedding vector: [x=[2,\ 4,\ 6,\ 8]]
- These are the token's four features because our embed_dim = 8 example is configurable; for easy calculation, we're using a vector of length 4 here.
- During training, feature values can vary in scale. RMSNorm rescales the vector to help keep its magnitude under control.
- Think of it as a volume controller for a vector: it adjusts the overall scale without subtracting the mean.


- RMSNorm formula:
- y = x / sqrt(mean(x^2) + eps)
- where eps is a small constant (e.g., 1e-5) to avoid division by zero.

- Step 1 : Square every value
-   x=[2,4,6,8]
- Step 2 : Calculate the mean of squares
-   4+16+36+64/4=30
- Step 3 : Calculate the RMS
- sqrt(30) aprox = 5.477
- Step 4: Divide each value by the RMS
- [2/5.477, 4/5.477, 6/5.477, 8/5.477] = [0.365, 0.730, 1.095, 1.461]
- The vector now has an RMS close to 1.

In [3]:
import torch
import torch.nn as nn

x = torch.tensor([2.0, 4.0, 6.0, 8.0])
x=x**2
print("Square of x:",x)

mean=x.mean()
print("Squared Mean:",mean)

rms=torch.sqrt(mean + 1e-8)
print("RMS:",rms)

normalized = x / rms
print("Normalized:", normalized)

Square of x: tensor([ 4., 16., 36., 64.])
Squared Mean: tensor(30.)
RMS: tensor(5.4772)
Normalized: tensor([ 0.7303,  2.9212,  6.5727, 11.6847])


In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, embed_dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        # Learnable scale parameter
        # gamma = weights
        self.gamma = nn.Parameter(torch.ones(embed_dim))
    def forward(self, x):
        # Square each feature
        squared_x = x ** 2
        # Mean of squared features
        mean_squared = squared_x.mean(dim=-1, keepdim=True) # Here, dim=-1 means that PyTorch calculates the mean across the last dimension, which represents the features.
        # Calculate RMS
        rms = torch.sqrt(mean_squared + self.eps)
        # Normalize the input
        normalized_x = x / rms
        # Apply learnable scaling
        output = normalized_x * self.gamma
        return output

# Test RMSNorm
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
model = RMSNorm(embed_dim=4)
output = model(x)
print("Input:", x)
print("Output:", output)
print("Output shape:", output.shape)

Input: tensor([[1., 2., 3., 4.]])
Output: tensor([[0.3651, 0.7303, 1.0954, 1.4606]], grad_fn=<MulBackward0>)
Output shape: torch.Size([1, 4])
